# Thực hành Lab 8


## Chuẩn bị
Chạy ô này trước, sau đó chạy các cell theo thứ tự. Nếu CSV ở nơi khác, điền đường dẫn vào các biến `DUONG_DAN_...`. Khi dùng Colab, tải CSV lên theo lời nhắc.

In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_absolute_error, silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import load_digits, load_sample_image
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC

# Để trống để tìm tự động; có thể chỉ định thư mục dữ liệu ở vị trí khác.
THU_MUC_DATA = os.environ.get("LAB8_DATA_DIR", "")
thu_muc_lab = Path(os.environ.get("LAB8_ROOT", Path.cwd())).resolve()
if "LAB8_ROOT" not in os.environ:
    if thu_muc_lab.name == "code": thu_muc_lab = thu_muc_lab.parent
    elif (thu_muc_lab / "08. K-Means").is_dir(): thu_muc_lab /= "08. K-Means"
thu_muc_hinh = thu_muc_lab / "figures"
thu_muc_hinh.mkdir(parents=True, exist_ok=True)

def tim_dataset(ten_tep):
    if THU_MUC_DATA:
        tep = Path(THU_MUC_DATA).expanduser() / ten_tep
        if tep.is_file(): return tep.resolve()
        raise FileNotFoundError(f"Khong tim thay {ten_tep} trong thu muc du lieu chi dinh.")
    for goc in [thu_muc_lab, Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        for tep in [goc / "08. K-Means" / "data" / ten_tep, goc / "data" / ten_tep, goc / ten_tep]:
            if tep.is_file(): return tep.resolve()
    if "google.colab" in sys.modules:
        from google.colab import files
        print(f"Hay tai len tep {ten_tep}.")
        files.upload()
        if Path(ten_tep).is_file(): return Path(ten_tep).resolve()
    raise FileNotFoundError(f"Can tep {ten_tep}; dat vao data hoac chi dinh THU_MUC_DATA.")

def doc_csv(ten_tep):
    tep = tim_dataset(ten_tep)
    print("Dataset dang su dung:", tep.name)
    return pd.read_csv(tep)

def luu_hinh(ten_tep):
    plt.gcf().tight_layout()
    plt.savefig(thu_muc_hinh / ten_tep, dpi=160, bbox_inches="tight")
    print(f"Da luu hinh: figures/{ten_tep}")
    plt.show()
    plt.close()

# Giữ nội dung cảnh báo hội tụ nhưng không in đường dẫn cài đặt cá nhân.
def canh_bao_ngan(message, category, filename, lineno, file=None, line=None):
    noi_dung = str(message)
    for goc in [thu_muc_lab, Path.home()]:
        for dang in [str(goc), goc.as_posix()]: noi_dung = noi_dung.replace(dang, "<thu_muc>")
    print(f"Canh bao {category.__name__}: {noi_dung}", file=file or sys.stdout)
warnings.showwarning = canh_bao_ngan

def chia_phan_loai(dac_trung, nhan):
    return train_test_split(dac_trung, nhan, test_size=0.25, random_state=17, stratify=nhan)


## Bước 1

In [2]:
# -*- coding: utf-8 -*-
"""Bước 1: tự viết K-Means bằng numpy, chạy trên bảy điểm của slide.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
from sklearn.cluster import KMeans

P = np.array([[ 1.0, 1.0], [ 1.5, 2.0], [ 3.0, 4.0], [ 5.0, 7.0],
[3.5, 5.0], [ 4.5, 5.0], [ 3.5, 4.5]])
TEN = [ "A", "B", "C", "D", "E", "F", "G"]


def k_means(X, tam, so_vong_toi_da= 10):
    """K-Means don gian: gan diem vao tam gan nhat roi doi tam."""
    for vong in range(1, so_vong_toi_da + 1):
        # Bước gán: khoảng cách từ mỗi điểm tới mỗi tâm
        kc = np.sqrt(((X[:, None, :] - tam[ None, :, :]) ** 2).sum(axis=2))
        nhan = kc.argmin(axis= 1)
        print(f"Vong {vong}: tam = {np.round(tam, 2).tolist()}" )
        for k in range(len(tam)):
            ds = [TEN[i] for i in range(len(X)) if nhan[i] == k]
            print(f" cum {k + 1}: {', '.join(ds)}" )
        # Bước dời tâm: trung bình các điểm trong mỗi cụm
        tam_moi = np.array([X[nhan == k].mean(axis= 0)
        for k in range(len(tam))])
        if np.allclose(tam_moi, tam):
            print(" Tam khong doi nua, dung." )
            return tam, nhan
        tam = tam_moi
    return tam, nhan


# Khởi tạo cố ý tệ: hai tâm là A và B, nằm sát nhau
tam, nhan = k_means(P, P[[ 0, 1]].copy())
print()

# Tính inertia: tổng bình phương khoảng cách tới tâm cụm của mình
inertia = ((P - tam[nhan]) ** 2).sum()
print(f"Inertia tu tinh: {inertia:.4f}" )

# So với scikit-learn, cho cùng tâm ban đầu
km = KMeans(n_clusters= 2, init=P[[ 0, 1]], n_init= 1).fit(P)
print(f"Inertia cua scikit-learn: {km.inertia_:.4f}" )
print("Tam cua scikit-learn:" , np. round(km.cluster_centers_, 2).tolist())


Vong 1: tam = [[1.0, 1.0], [1.5, 2.0]]
 cum 1: A
 cum 2: B, C, D, E, F, G
Vong 2: tam = [[1.0, 1.0], [3.5, 4.58]]
 cum 1: A, B
 cum 2: C, D, E, F, G
Vong 3: tam = [[1.25, 1.5], [3.9, 5.1]]
 cum 1: A, B
 cum 2: C, D, E, F, G
 Tam khong doi nua, dung.

Inertia tu tinh: 8.5250


Inertia cua scikit-learn: 8.5250
Tam cua scikit-learn: [[1.25, 1.5], [3.9, 5.1]]


## Bước 2

In [3]:
# -*- coding: utf-8 -*-
"""Bước 2: phân khúc khách hàng bằng K-Means."""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(tim_dataset("khach_hang.csv"))
print("Kich thuoc bang:" , df.shape)
print(df.describe().round(1).to_string())
print()

X = df[[ "thu_nhap", "diem_chi_tieu"]]
sc = StandardScaler().fit(X)
Xs = sc.transform(X)

km = KMeans(n_clusters= 5, n_init= 10, random_state= 0).fit(Xs)
df["cum"] = km.labels_

# Tâm cụm đang ở thang đã chuẩn hoá, đổi ngược về đơn vị gốc để đọc
tam = pd.DataFrame(sc.inverse_transform(km.cluster_centers_),
columns=["thu_nhap", "diem_chi_tieu"])
tam["so_khach"] = np.bincount(km.labels_)
tam["tuoi_tb"] = df.groupby( "cum")["tuoi"].mean()
print("Tam cum, don vi goc:" )
print(tam.round(1).to_string())
print()
print(f"Inertia: {km.inertia_:.2f}" )
print()

# Khách mới: chuẩn hoá bằng đúng bộ chuẩn hoá cũ rồi tìm tâm gần nhất
moi = pd.DataFrame({ "thu_nhap": [ 29.0, 9.0],
"diem_chi_tieu": [ 85, 75]})
cum_moi = km.predict(sc.transform(moi))
for i in range(len(moi)):
    print(f"Khach moi {i + 1}: thu nhap {moi.iloc[i, 0]},"
    f" chi tieu {moi.iloc[i, 1]} -> cum {cum_moi[i]}" )


Kich thuoc bang: (210, 3)
        tuoi  thu_nhap  diem_chi_tieu
count  210.0     210.0          210.0
mean    37.6      18.5           49.9
std     10.3       9.8           26.8
min     18.0       3.0            7.0
25%     30.0       8.8           22.0
50%     39.0      17.4           49.0
75%     46.0      28.6           77.0
max     62.0      35.5           99.0

Tam cum, don vi goc:
   thu_nhap  diem_chi_tieu  so_khach  tuoi_tb
0       7.5           80.2        35     24.9
1      30.1           20.0        35     42.9
2      17.4           49.3        60     40.4
3       8.0           18.9        40     44.9
4      30.2           81.5        40     32.6

Inertia: 23.02

Khach moi 1: thu nhap 29.0, chi tieu 85 -> cum 4
Khach moi 2: thu nhap 9.0, chi tieu 75 -> cum 0


## Bước 3

In [4]:
# -*- coding: utf-8 -*-
"""Bước 3: chọn số cụm K bằng khuỷu tay và chỉ số silhouette."""

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(tim_dataset("khach_hang.csv"))
Xs = StandardScaler().fit_transform(df[[ "thu_nhap", "diem_chi_tieu"]])

print(" K inertia giam so voi K-1 silhouette" )
truoc = None
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init= 10, random_state= 0).fit(Xs)
    giam = "" if truoc is None else f"{truoc - km.inertia_:8.1f}"
    # Silhouette cần ít nhất hai cụm
    sil = "" if k == 1 else f"{silhouette_score(Xs, km.labels_):.3f}"
    print(f" {k:2d} {km.inertia_:7.1f} {giam:>15s} {sil:>10s}" )
    truoc = km.inertia_


 K inertia giam so voi K-1 silhouette
  1   420.0                           
  2   250.4           169.6      0.430
  3   152.5            97.9      0.522
  4    73.8            78.8      0.626
  5    23.0            50.7      0.723
  6    20.9             2.1      0.651
  7    19.0             1.9      0.541


  8    17.4             1.6      0.454
  9    15.9             1.5      0.471
 10    14.4             1.4      0.425


## Bước 4

In [5]:
# -*- coding: utf-8 -*-
"""Bước 4: ba cái bẫy của K-Means."""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(tim_dataset("khach_hang.csv"))
X = df[[ "thu_nhap", "diem_chi_tieu"]]
Xs = StandardScaler().fit_transform(X)

# Bẫy 1: thu nhập ghi bằng nghìn đồng, không chuẩn hoá
X_nghin = X.copy()
X_nghin["thu_nhap"] = X_nghin[ "thu_nhap"] * 1000
km_tho = KMeans(n_clusters= 5, n_init= 10, random_state= 0).fit(X_nghin)
tam_tho = km_tho.cluster_centers_.copy()
tam_tho[:, 0] = tam_tho[:, 0] / 1000 # đổi lại triệu đồng để dễ đọc
print("Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):" )
for t in tam_tho[np.argsort(tam_tho[:, 0])]:
    print(f" ({t[0]:5.1f}, {t[1]:5.1f})" )
print()

# Bẫy 2: mỗi lần chỉ khởi tạo ngẫu nhiên một lần
ket = []
for rs in range(20):
    km = KMeans(n_clusters= 5, init= "random", n_init= 1,
    random_state=rs).fit(Xs)
    ket.append(round(km.inertia_, 1))
print("Bay 2: inertia cua 20 lan khoi tao ngau nhien:" )
ket = sorted(ket)
print(" " , ket[: 10])
print(" " , ket[ 10:])
tot = KMeans(n_clusters= 5, n_init= 10, random_state= 0).fit(Xs)
print(f" Voi n_init=10: inertia {tot.inertia_:.1f}" )
print()

# Bẫy 3: cụm không tròn, dữ liệu hai vầng trăng
tr = pd.read_csv(tim_dataset("hai_vang_trang.csv"))
km_tr = KMeans(n_clusters= 2, n_init= 10, random_state= 0)
nhan = km_tr.fit_predict(tr[[ "x1", "x2"]])
khop = (nhan == tr[ "nhan"]).mean()
# Số thứ tự cụm có thể ngược với nhãn thật, nên lấy cách khớp tốt hơn
print(f"Bay 3: hai vang trang, ty le khop nhan that "
f"{max(khop, 1 - khop):.4f}" )


Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):
 (  6.1,  52.5)
 (  9.6,  41.9)
 ( 17.4,  49.8)
 ( 27.8,  52.9)
 ( 31.9,  52.4)

Bay 2: inertia cua 20 lan khoi tao ngau nhien:
  [23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0]
  [72.4, 73.6, 73.7, 73.7, 84.0, 84.2, 84.2, 116.6, 117.4, 118.2]
 Voi n_init=10: inertia 23.0

Bay 3: hai vang trang, ty le khop nhan that 0.7500


## Bước 5

In [7]:
# -*- coding: utf-8 -*-
"""Bước 5: nén màu một bức ảnh bằng K-Means."""

import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image

anh = load_sample_image( "china.jpg") # ảnh có sẵn trong scikit-learn
cao, rong, _ = anh.shape
diem = anh.reshape(- 1, 3) / 255.0 # mỗi dòng là một điểm ảnh RGB
so_mau = len(np.unique(anh.reshape(-1, 3), axis= 0))
print(f"Anh {cao} x {rong}, co {cao * rong} diem anh, {so_mau} mau" )
print()

# Học trên 5000 điểm ảnh rút ngẫu nhiên cho nhanh, rồi gán cho mọi điểm
rng = np.random.default_rng( 0)
mau = diem[rng.choice( len(diem), 5000, replace= False)]

print(" K so bit moi diem nho hon sai lech trung binh" )
for k in [4, 16, 64]:
    km = KMeans(n_clusters=k, n_init= 4, random_state= 0).fit(mau)
    nhan = km.predict(diem)
    moi = km.cluster_centers_[nhan]
    bit = int(np.ceil(np.log2(k)))
    sai = np. abs(moi - diem).mean() * 255
    print(f" {k:2d} {bit} {24 / bit:4.1f} lan"
    f" {sai:5.2f} muc sang" )


Anh 427 x 640, co 273280 diem anh, 96615 mau

 K so bit moi diem nho hon sai lech trung binh
  4 2 12.0 lan 17.41 muc sang
 16 4  6.0 lan  7.97 muc sang
 64 6  4.0 lan  4.54 muc sang
